# Task 1.3 — Ingest IoT Passenger Counter Data

- 1.4M rows/day from ~1,000 sensors — partition by `terminal` + `event_date`
- 60-second micro-batch streaming from sensor network
- DQ: `queue_length_estimate` must not be negative; `avg_wait_time_minutes` must be < 180
- Schema evolution: sensor firmware updates may add new fields — use `mergeSchema=True`
- Store raw sensor payload as JSON string in Bronze


In [ ]:
# ── Imports & Configuration ────────────────────────────────────────
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

DATA_DIR = "/FileStore/airport_analytics_data"
BRONZE_DIR = "/FileStore/delta_lake/bronze"

try:
    spark
except NameError:
    spark = (SparkSession.builder
        .appName("Task_1_3_IoT_Counters_Bronze")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog",
                "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate())


## Step 1 — Raw Ingestion

In [ ]:
# ── Raw Ingestion: passenger_counters.json (NDJSON) ──────────────
raw_iot_df = spark.read.json(
    f"{DATA_DIR}/passenger_counters/passenger_counters.json"
)

# Store raw sensor payload as JSON string for schema evolution safety
raw_iot_df = (raw_iot_df
    .withColumn("raw_payload", F.to_json(F.struct("*")))
    .withColumn("ingestion_ts", F.current_timestamp())
    .withColumn("event_date", F.to_date(F.col("timestamp")))
)

print(f"Raw IoT records ingested: {raw_iot_df.count()}")
raw_iot_df.printSchema()
raw_iot_df.show(5, truncate=False)


## Step 2 — Data Quality & Sanity Checks

In [ ]:
# ── DQ Checks ────────────────────────────────────────────────────
# 1. queue_length_estimate must not be negative
neg_queue = raw_iot_df.filter(F.col("queue_length_estimate") < 0)
print(f"Negative queue_length_estimate: {neg_queue.count()}")

# 2. avg_wait_time_minutes must be < 180
extreme_wait = raw_iot_df.filter(F.col("avg_wait_time_minutes") >= 180)
print(f"avg_wait_time_minutes >= 180: {extreme_wait.count()}")

# 3. sensor_id should not be null
null_sensor = raw_iot_df.filter(F.col("sensor_id").isNull())
print(f"Null sensor_id records: {null_sensor.count()}")

# Build validated dataframe
validated_iot_df = (raw_iot_df
    .filter(F.col("queue_length_estimate") >= 0)
    .filter(F.col("avg_wait_time_minutes") < 180)
    .filter(F.col("sensor_id").isNotNull())
    .withColumn("timestamp", F.to_timestamp("timestamp"))
)
print(f"\nValidated IoT records: {validated_iot_df.count()}")
validated_iot_df.show(5, truncate=False)


## Step 3 — Write to Bronze Delta Table

In [ ]:
# ── Write to Bronze Delta (append, partition by terminal + event_date)
bronze_iot_path = f"{BRONZE_DIR}/iot_counters"

(validated_iot_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .partitionBy("terminal", "event_date")
    .save(bronze_iot_path))

print(f"IoT counters written to {bronze_iot_path}")
bronze_iot_df = spark.read.format("delta").load(bronze_iot_path)
print(f"Bronze iot_counters total rows: {bronze_iot_df.count()}")
print(f"Partitions: {bronze_iot_df.select('terminal','event_date').distinct().count()}")
bronze_iot_df.show(5, truncate=False)
